<a href="https://colab.research.google.com/github/adib422/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I will basically be using model for answering yes/no for each page — "does this page need attention?" — and then using that yes/no score to build a priority list. So: it's a classification problem, used to make a ranking

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is is_worth_reviewing: a page is flagged 1 if trend_direction is "down" AND impressions_90d is at least 100. This is an observed outcome pulled from real measured signals in the data, not a rule I invented myself — which matters, because a model trained on someone's hand-written rule just re-learns that rule instead of learning anything real about the world. One caution I'm noting for later: I need to check the leakage skill to confirm trend_direction is measured over a window that doesn't overlap with my predictor features, or the target could leak into the inputs.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

I'll measure success with Precision@50 on a client-holdout split: of the top 50 pages my model flags, how many were actually genuinely worth reviewing. I'm not using overall accuracy, because an editor only has time to check a limited number of pages each week — being right near the top of the list matters far more than being right everywhere. This also matches what the starter pipeline already validated (baseline rule 0.24 vs. random forest 0.74).

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Build the target column
df["is_worth_reviewing"] = (
    df["trend_direction"].str.lower().eq("down") & (df["impressions_90d"] >= 100)
)

# One row = one content page, scored on its trailing 90-day signals
cols = ["content_id", "trend_direction", "impressions_90d", "is_worth_reviewing"]
df[cols].head(10)

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


,content_id,trend_direction,impressions_90d,is_worth_reviewing
0,content_304f48230142,down,3803,True
1,content_a1fb4e703a9e,down,15320,True
2,content_9aa793d4d895,down,12581,True
3,content_331d6c4de07b,stable,11751,False
4,content_d99b7a2d90ca,down,19140,True
5,content_d4084a4bc775,down,3970,True
6,content_9a34b442b552,down,20,False
7,content_a63219c6e95a,stable,1724,False
8,content_5e6c160719bc,down,32574,True
9,content_c27558df2b0c,down,1240,True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Whether a page is worth reviewing depends on several signals moving together — staleness, visibility, ranking position, click-through rate — and how they interact changes by content type and traffic tier. That's too tangled to capture confidently in a single if-else rule. I already have proof from the starter pipeline: a simple trained model roughly tripled precision@50 over the hand-written rule (0.24 → 0.74) on this exact dataset.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.